In [1]:
import numpy as np
import pandas as pd

In [2]:
# Load the data
df = pd.read_csv('depthDep_ADCPdata.csv')
df.head()

C:\Users\Hosse\AppData\Local\Temp\ipykernel_19764\2495231642.py:2: DtypeWarning: Columns (2,3,4,6,7,8,10,11,12,14,15,16,18,19,20,22,23,24,26,27,28,30,31,32,34,35,36,38,39,40,42,43,44,46,47,48,50,51,52,54,55,56,58,59,60,62,63,64,66,67,68,70,71,72,74,75,76,78,79,80,82,83,84,86,87,88,90,91,92,94,95,96,98,99,100,102,103,104,106,107,108,110,111,112,114,115,116,118,119,120,122,123,124,126,127,128,130,131,132,134,135,136,138,139,140,142,143,144,146,147,148,150,151,152,154,155,156,158,159,160,162,163,164,166,167,168,170,171,172,174,175,176,178,179,180,182,183,184,186,187,188,190,191,192,194,195,196) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('depthDep_ADCPdata.csv')


,DateTime,Bin #:,1,1.1,1.2,1.3,2,2.1,2.2,2.3,...,47.2,47.3,48,48.1,48.2,48.3,49,49.1,49.2,49.3
0,NaN,NaN,Eas,Nor,Mag,NaN,Eas,Nor,Mag,NaN,...,Mag,NaN,Eas,Nor,Mag,NaN,Eas,Nor,Mag,NaN
1,2024-10-23 09:00:00,NaN,11,-7,13,117.47,12,-6,13,111.47,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-10-23 09:10:00,NaN,-88,-55,104,232.47,-82,-52,97,232.47,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-10-23 09:20:00,NaN,-181,-67,193,244.47,-171,-84,191,238.47,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-10-23 09:30:00,NaN,-243,-61,251,250.47,-246,-46,250,253.47,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Inspect Depth-Averaged Data Structure
This cell loads and examines the depth-averaged ADCP data from `depthAvg_ADCPdata_labeled.csv` to understand the data structure, column names, and available velocity components. This file contains pre-processed depth-averaged velocities with tidal phase classifications.

In [3]:
# Load the depth-averaged data to examine velocity components structure
depth_avg_df = pd.read_csv('depthAvg_ADCPdata_labeled.csv')
print("Depth-averaged data columns:")
print(depth_avg_df.columns.tolist())
print(f"\nData shape: {depth_avg_df.shape}")
print("\nFirst few rows:")
print(depth_avg_df.head())

Depth-averaged data columns:
['Date & Time.2', 'Eas', 'Nor', 'Mag', 'Dir', 'theta_used_degT', 'u_along_overall', 'phase']

Data shape: (16544, 8)

First few rows:
         Date & Time.2    Eas    Nor    Mag     Dir  theta_used_degT  \
0  2024-10-23 09:00:00  0.185  0.069  0.197   64.47            64.47   
1  2024-10-23 09:10:00  0.091  0.046  0.102   57.47            57.47   
2  2024-10-23 09:20:00  0.013  0.034  0.036   14.47            14.47   
3  2024-10-23 09:30:00 -0.079  0.009  0.080  271.47           271.47   
4  2024-10-23 09:40:00 -0.175 -0.001  0.175  264.47           264.47   

   u_along_overall  phase  
0         0.196537  slack  
1         0.099785  slack  
2         0.021329  slack  
3        -0.074006  slack  
4        -0.169338  slack  


## Verify Along-Channel and Across-Channel Components
This cell checks if the depth-averaged data already contains the along-channel (`u_along`) and across-channel (`u_across`) velocity components that have been rotated using the principal axis transformation. These components are essential for the coordinate system used in this analysis.

In [4]:
# Check if along/across velocity components exist in depth-averaged data
if 'u_along' in depth_avg_df.columns and 'u_across' in depth_avg_df.columns:
    print("Found u_along and u_across in depth-averaged data!")
    print(f"u_along range: {depth_avg_df['u_along'].min():.3f} to {depth_avg_df['u_along'].max():.3f}")
    print(f"u_across range: {depth_avg_df['u_across'].min():.3f} to {depth_avg_df['u_across'].max():.3f}")

# Check data types
print("\nDataFrame data types:")
print(depth_avg_df.dtypes)
print(f"\nTime column type: {type(depth_avg_df['Date & Time.2'].iloc[0])}")
print(f"Sample time values: {depth_avg_df['Date & Time.2'].head()}")


DataFrame data types:
Date & Time.2       object
Eas                float64
Nor                float64
Mag                float64
Dir                float64
theta_used_degT    float64
u_along_overall    float64
phase               object
dtype: object

Time column type: <class 'str'>
Sample time values: 0    2024-10-23 09:00:00
1    2024-10-23 09:10:00
2    2024-10-23 09:20:00
3    2024-10-23 09:30:00
4    2024-10-23 09:40:00
Name: Date & Time.2, dtype: object


## Parse DateTime Column
This cell converts the `DateTime` column in the depth-dependent data from string format to pandas datetime objects. This is essential for time-based filtering and analysis of tidal cycles in subsequent steps.

In [5]:
# Parse datetime
df['DateTime'] = pd.to_datetime(df['DateTime'])
df.head()

,DateTime,Bin #:,1,1.1,1.2,1.3,2,2.1,2.2,2.3,...,47.2,47.3,48,48.1,48.2,48.3,49,49.1,49.2,49.3
0,NaT,NaN,Eas,Nor,Mag,NaN,Eas,Nor,Mag,NaN,...,Mag,NaN,Eas,Nor,Mag,NaN,Eas,Nor,Mag,NaN
1,2024-10-23 09:00:00,NaN,11,-7,13,117.47,12,-6,13,111.47,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-10-23 09:10:00,NaN,-88,-55,104,232.47,-82,-52,97,232.47,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-10-23 09:20:00,NaN,-181,-67,193,244.47,-171,-84,191,238.47,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-10-23 09:30:00,NaN,-243,-61,251,250.47,-246,-46,250,253.47,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Extract Depth Bin Information
This cell identifies and extracts the column names corresponding to different depth bins in the ADCP data. The data structure uses every 4th column starting from column 2 to represent different depth measurements, allowing us to understand the depth-resolved velocity structure.

In [6]:
# Extract bin numbers (every 4th column starting from column 2)
bin_cols = []
for i in range(2, len(df.columns), 4):  # Start at column 2, step by 4
    if i < len(df.columns):
        bin_cols.append(df.columns[i])

print(f"Found {len(bin_cols)} depth bins")
print(f"Bin columns: {bin_cols[:10]}...")  # Show first 10

Found 49 depth bins
Bin columns: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']...


## Create Depth Mapping for ADCP Bins
This cell creates a mapping between ADCP bin numbers and their corresponding depths in meters. The ADCP configuration starts at 1.67m depth with 0.5m intervals between bins. The analysis uses bins 1-44 (1.67m to 23.17m depth) while excluding bins 45-49 due to significant data deviation near the bottom.

In [7]:
# Create bin-to-depth mapping
def bin_to_depth(bin_number):
    """Convert bin number to depth in meters"""
    return 1.67 + (bin_number - 1) * 0.5

# Create depth mapping dictionary
depth_mapping = {}
for bin_num in range(1, 50):  # Bins 1 to 49 (keeping full mapping for reference)
    depth_mapping[bin_num] = bin_to_depth(bin_num)

print("Depth mapping (first 10 bins):")
for bin_num in range(1, 11):
    print(f"Bin {bin_num}: {depth_mapping[bin_num]:.2f} m")

print(f"...")
print(f"Bin 44: {depth_mapping[44]:.2f} m (last bin used)")
print(f"Bin 45-49: {depth_mapping[45]:.2f}-{depth_mapping[49]:.2f} m (excluded due to significant deviation)")

Depth mapping (first 10 bins):
Bin 1: 1.67 m
Bin 2: 2.17 m
Bin 3: 2.67 m
Bin 4: 3.17 m
Bin 5: 3.67 m
Bin 6: 4.17 m
Bin 7: 4.67 m
Bin 8: 5.17 m
Bin 9: 5.67 m
Bin 10: 6.17 m
...
Bin 44: 23.17 m (last bin used)
Bin 45-49: 23.67-25.67 m (excluded due to significant deviation)


## Define Channel Rotation Angle Function
This cell defines the essential function for coordinate transformation:

**`get_channel_rotation_angle()`**: Returns the predetermined principal axis angle (75.3°T) from the ebbFloodClassifier.py analysis, eliminating the need for dynamic angle calculation. This angle aligns with the principal flow direction determined by principal component analysis.

In [8]:
def get_channel_rotation_angle(depth_avg_df):
    """
    Use predetermined channel rotation angle from ebbFloodClassifier.py analysis
    
    The principal axis angles are:
    - Ebb direction: 75.36°T (toward ocean)
    - Flood direction: 255.36°T (toward Long Island Sound)
    
    Returns the angle in degrees for east-north to along-across coordinate transformation
    """
    # Use the ebb direction as the primary channel axis (75.3°T)
    # This aligns with the principal flow direction determined by ebbFloodClassifier.py
    principal_axis_angle = 255.36  # degrees True (geographic)
    
    print(f"Using predetermined channel rotation angle: {principal_axis_angle:.1f}°T")
    print("This angle is based on principal axis analysis from ebbFloodClassifier.py:")
    print(f"  • Ebb direction (toward ocean): {principal_axis_angle:.1f}°T")
    print(f"  • Flood direction (toward LIS): {principal_axis_angle + 180:.1f}°T")
    
    return principal_axis_angle

## Load Tidal Phase Data and Identify Individual Cycles
This cell contains functions to process tidal phase classifications and identify individual ebb and flood cycles:

1. **`load_tidal_phases()`**: Loads the labeled tidal phase data with datetime parsing
2. **`identify_tidal_cycles()`**: Processes the phase data to identify individual tidal cycles by detecting phase transitions (EBB → FLOOD → EBB), excluding SLACK periods, and calculating cycle durations

The execution identifies all tidal cycles in the dataset, providing temporal boundaries for cycle-based turbulence intensity analysis.

In [11]:
# Load the tidal phase data
def load_tidal_phases():
    """Load and process the labeled tidal phase data"""
    phase_df = pd.read_csv('depthAvg_ADCPdata_labeled.csv')
    phase_df['Time'] = pd.to_datetime(phase_df['Date & Time.2'], format='mixed', errors='coerce')
    print(f"Successfully loaded depth-averaged data: {phase_df.shape}")
    print(f"Available columns: {phase_df.columns.tolist()}")
    return phase_df

def identify_tidal_cycles(phase_df):
    """
    Identify individual ebb and flood cycles from the phase data
    Returns a list of cycle dictionaries with start/end times and cycle type
    """
    cycles = []
    current_phase = None
    cycle_start = None
    cycle_id = 0
    
    for i, row in phase_df.iterrows():
        phase = row['phase']
        timestamp = row['Time']
        
        # Skip SLACK periods for cycle identification
        if phase == 'SLACK':
            continue
            
        # Check if phase has changed (new cycle starts)
        if phase != current_phase:
            # End previous cycle if it exists
            if current_phase is not None and cycle_start is not None:
                cycles.append({
                    'cycle_id': cycle_id,
                    'phase': current_phase,
                    'start_time': cycle_start,
                    'end_time': prev_timestamp,
                    'duration_hours': (prev_timestamp - cycle_start).total_seconds() / 3600
                })
                cycle_id += 1
            
            # Start new cycle
            current_phase = phase
            cycle_start = timestamp
        
        prev_timestamp = timestamp
    
    # Add the last cycle
    if current_phase is not None and cycle_start is not None:
        cycles.append({
            'cycle_id': cycle_id,
            'phase': current_phase,
            'start_time': cycle_start,
            'end_time': prev_timestamp,
            'duration_hours': (prev_timestamp - cycle_start).total_seconds() / 3600
        })
    
    return pd.DataFrame(cycles)

# Load phase data and identify cycles
print("Loading tidal phase data...")
phase_df = load_tidal_phases()

print("Identifying tidal cycles...")
cycles_df = identify_tidal_cycles(phase_df)

print(f"\nFound {len(cycles_df)} tidal cycles:")
print(f"  Ebb cycles: {len(cycles_df[cycles_df['phase'] == 'EBB'])}")
print(f"  Flood cycles: {len(cycles_df[cycles_df['phase'] == 'FLOOD'])}")

print("\nFirst 10 cycles:")
print(cycles_df.head(10))

Loading tidal phase data...
Successfully loaded depth-averaged data: (16544, 9)
Available columns: ['Date & Time.2', 'Eas', 'Nor', 'Mag', 'Dir', 'theta_used_degT', 'u_along_overall', 'phase', 'Time']
Identifying tidal cycles...

Found 891 tidal cycles:
  Ebb cycles: 0
  Flood cycles: 0

First 10 cycles:
   cycle_id  phase          start_time            end_time  duration_hours
0         0  slack 2024-10-23 09:00:00 2024-10-23 10:00:00        1.000000
1         1  flood 2024-10-23 10:10:00 2024-10-23 15:10:00        5.000000
2         2  slack 2024-10-23 15:20:00 2024-10-23 16:30:00        1.166667
3         3    ebb 2024-10-23 16:40:00 2024-10-23 21:20:00        4.666667
4         4  slack 2024-10-23 21:30:00 2024-10-23 23:00:00        1.500000
5         5  flood 2024-10-23 23:10:00 2024-10-24 03:50:00        4.666667
6         6  slack 2024-10-24 04:00:00 2024-10-24 05:20:00        1.333333
7         7    ebb 2024-10-24 05:30:00 2024-10-24 09:30:00        4.000000
8         8  slack 2

## Export Depth-Dependent Velocity Components
This cell contains a function to process and export the along-channel and across-channel velocity components for all depth bins in the depth-dependent ADCP data:

**`export_depth_dependent_velocity_components()`**: 
- Transforms East-North velocities to along-channel/across-channel coordinates for all 44 depth bins
- Uses the predetermined 75.3°T rotation angle from ebbFloodClassifier.py
- Exports comprehensive dataset with DateTime, depth information, and transformed velocity components
- Creates `depth_dependent_velocity_components.csv` for further analysis

This export provides the foundation for detailed depth-resolved turbulence analysis and allows external validation of the coordinate transformation approach.

In [12]:
def export_depth_dependent_velocity_components(df, depth_avg_df=None, max_bin=44, output_filename='depth_dependent_velocity_components.csv'):
    """
    Export along-channel and across-channel velocity components for all depth bins
    
    Parameters:
    - df: DataFrame with depth-dependent ADCP data
    - depth_avg_df: DataFrame for coordinate transformation angle (optional)
    - max_bin: Maximum bin number to process (default: 44)
    - output_filename: Name of output CSV file
    
    Returns:
    - DataFrame with transformed velocity components
    """
    print("="*80)
    print("EXPORTING DEPTH-DEPENDENT VELOCITY COMPONENTS")
    print("Converting East-North to Along-Channel/Across-Channel coordinates")
    print("="*80)
    
    # Get channel rotation angle
    channel_angle = None
    if depth_avg_df is not None:
        channel_angle = get_channel_rotation_angle(depth_avg_df)
        print(f"Using channel rotation angle: {channel_angle:.2f}°T")
    else:
        # Use default angle if no depth_avg_df provided
        channel_angle = 75
        print(f"Using default channel rotation angle: {channel_angle:.2f}°T")
    
    # Prepare results list
    results = []
    
    # Process each depth bin (outer loop)
    print(f"Processing {max_bin} depth bins across {len(df)} time steps...")
    
    for bin_num in range(1, max_bin + 1):
        print(f"Processing bin {bin_num} (depth: {depth_mapping[bin_num]:.2f} m)...")
        
        # Find columns for this bin
        eas_col = f"{bin_num}"          # East velocity
        nor_col = f"{bin_num}.1"        # North velocity
        
        # Check if columns exist
        if eas_col not in df.columns or nor_col not in df.columns:
            print(f"  Warning: Columns for bin {bin_num} not found, skipping...")
            continue
        
        # Process each time step for this bin
        for idx, row in df.iterrows():
            datetime_stamp = row['DateTime']
                                 
            # Extract velocity components (convert from mm/s to m/s)
            u_east = pd.to_numeric(row[eas_col], errors='coerce') / 1000
            v_north = pd.to_numeric(row[nor_col], errors='coerce') / 1000
            
            # Skip if either velocity is NaN
            if pd.isna(u_east) or pd.isna(v_north):
                continue
            
            # Apply coordinate transformation
            theta = np.radians(channel_angle)
            u_along = u_east * np.sin(theta) + v_north * np.cos(theta)    # along-channel
            u_across = u_east * np.cos(theta) - v_north * np.sin(theta)  # across-channel

            # Store results
            result_row = {
                'DateTime': datetime_stamp,
                'bin_number': bin_num,
                'depth_m': depth_mapping[bin_num],
                'u_east_ms': u_east,
                'v_north_ms': v_north,
                'u_along_ms': u_along,
                'u_across_ms': u_across,
                'channel_angle_deg': channel_angle
            }
            
            results.append(result_row)
        
        # Progress indicator for this bin
        bin_data_count = len([r for r in results if r['bin_number'] == bin_num])
        print(f"  Completed bin {bin_num}: {bin_data_count:,} valid data points")
    
    # Create DataFrame
    velocity_df = pd.DataFrame(results)
    
    if len(velocity_df) > 0:
        print(f"\\nSuccessfully processed {len(velocity_df):,} velocity measurements")
        print(f"Time range: {velocity_df['DateTime'].min()} to {velocity_df['DateTime'].max()}")
        print(f"Depth range: {velocity_df['depth_m'].min():.2f} to {velocity_df['depth_m'].max():.2f} m")
        print(f"Number of depth bins: {velocity_df['bin_number'].nunique()}")
        
        # Classify tidal phases (EBB, FLOOD, SLACK)
        print(f"\\nClassifying tidal phases based on along-channel velocity...")
        velocity_threshold = 0.05  # m/s threshold for phase classification
        
        # Use a representative depth bin (middle of water column) for phase classification
        representative_bin = velocity_df['bin_number'].unique()[len(velocity_df['bin_number'].unique()) // 2]
        print(f"Using bin {representative_bin} (depth: {depth_mapping[representative_bin]:.2f} m) for phase classification")
        
        # Get along-channel velocity for representative bin, sorted by time
        phase_data = velocity_df[velocity_df['bin_number'] == representative_bin].sort_values('DateTime').copy()
        
        # Apply smoothing to reduce noise (3-point moving average)
        phase_data['u_along_smooth'] = phase_data['u_along_ms'].rolling(window=3, center=True, min_periods=1).mean()
        
        # Classify phases
        def classify_phase(velocity):
            if velocity > velocity_threshold:
                return "FLOOD"
            elif velocity < -velocity_threshold:
                return "EBB"
            else:
                return "SLACK"
        
        phase_data['phase'] = phase_data['u_along_smooth'].apply(classify_phase)
        
        # Create a mapping from DateTime to phase
        phase_mapping = dict(zip(phase_data['DateTime'], phase_data['phase']))
        
        # Apply phase classification to all bins
        velocity_df['phase'] = velocity_df['DateTime'].map(phase_mapping)
        
        # Fill any remaining NaN phases with SLACK
        velocity_df['phase'].fillna('SLACK', inplace=True)
        
        # Print phase statistics
        phase_counts = velocity_df.groupby('phase').size()
        total_records = len(velocity_df)
        print(f"\\nPhase classification summary:")
        for phase in ['FLOOD', 'EBB', 'SLACK']:
            if phase in phase_counts.index:
                count = phase_counts[phase]
                print(f"  {phase}: {count:,} records ({count/total_records*100:.1f}%)")
        
        # Calculate summary statistics
        print(f"\\nVelocity component statistics [m/s]:")
        print(f"  u_along  (along-channel):  {velocity_df['u_along_ms'].mean():.4f} ± {velocity_df['u_along_ms'].std():.4f}")
        print(f"                             Range: {velocity_df['u_along_ms'].min():.4f} to {velocity_df['u_along_ms'].max():.4f}")
        print(f"  u_across (across-channel): {velocity_df['u_across_ms'].mean():.4f} ± {velocity_df['u_across_ms'].std():.4f}")
        print(f"                             Range: {velocity_df['u_across_ms'].min():.4f} to {velocity_df['u_across_ms'].max():.4f}")
        
        # Phase-specific velocity statistics
        print(f"\\nPhase-specific along-channel velocity statistics [m/s]:")
        for phase in ['FLOOD', 'EBB']:
            phase_data = velocity_df[velocity_df['phase'] == phase]
            if len(phase_data) > 0:
                print(f"  {phase}: {phase_data['u_along_ms'].mean():.4f} ± {phase_data['u_along_ms'].std():.4f}")
                print(f"         Range: {phase_data['u_along_ms'].min():.4f} to {phase_data['u_along_ms'].max():.4f}")
        
        # Export to CSV
        velocity_df.to_csv(output_filename, index=False)
        print(f"\\nExported velocity components to: {output_filename}")
        print(f"File contains {len(velocity_df):,} records with {len(velocity_df.columns)} columns")
        
        # Show column information
        print(f"\\nColumn information:")
        for col in velocity_df.columns:
            print(f"  {col}")
        
        print(f"\\nFirst few records:")
        print(velocity_df.head())
        
        return velocity_df
    else:
        print("\\nNo valid velocity data found!")
        return pd.DataFrame()

# Execute the export function
print("\\nExporting depth-dependent velocity components...")

# Load depth-averaged data if not already loaded (for channel angle)
if 'depth_avg_df' not in locals() or depth_avg_df is None:
    try:
        depth_avg_df = pd.read_csv('depthAvg_ADCPdata_labeled.csv')
        print(f"Loaded depth-averaged data for coordinate transformation: {depth_avg_df.shape}")
    except FileNotFoundError:
        print("Warning: depthAvg_ADCPdata_labeled.csv not found. Using default angle.")
        depth_avg_df = None

# Export velocity components
depth_velocity_df = export_depth_dependent_velocity_components(df, depth_avg_df)

print(f"\\nVelocity component export completed!")
print(f"Results available in DataFrame: depth_velocity_df")
print(f"CSV file: depth_dependent_velocity_components.csv")

\nExporting depth-dependent velocity components...
EXPORTING DEPTH-DEPENDENT VELOCITY COMPONENTS
Converting East-North to Along-Channel/Across-Channel coordinates
Using predetermined channel rotation angle: 255.4°T
This angle is based on principal axis analysis from ebbFloodClassifier.py:
  • Ebb direction (toward ocean): 255.4°T
  • Flood direction (toward LIS): 435.4°T
Using channel rotation angle: 255.36°T
Processing 44 depth bins across 16543 time steps...
Processing bin 1 (depth: 1.67 m)...
  Completed bin 1: 16,542 valid data points
Processing bin 2 (depth: 2.17 m)...
  Completed bin 2: 16,542 valid data points
Processing bin 3 (depth: 2.67 m)...
  Completed bin 3: 16,542 valid data points
Processing bin 4 (depth: 3.17 m)...
  Completed bin 4: 16,542 valid data points
Processing bin 5 (depth: 3.67 m)...
  Completed bin 5: 16,542 valid data points
Processing bin 6 (depth: 4.17 m)...
  Completed bin 6: 16,542 valid data points
Processing bin 7 (depth: 4.67 m)...
  Completed bin 7: 